# 01 · Exploração dos Dados (EDA)

Entendemos o dataset antes de qualquer modelagem.
Descobrir padrões aqui guia todas as decisões seguintes.


## Setup


In [ ]:
# Adiciona a raiz do projeto ao path para importar src/
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_raw
from src.visualization.plots import plot_class_distribution


ModuleNotFoundError: No module named 'seaborn'

## 1. Carregamento


In [ ]:
df = load_raw()
df.head()


## 2. Visão Geral


In [ ]:
print(f'Shape: {df.shape}')
print(f'Valores nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.describe().round(3)


## 3. Distribuição das Classes

Visualizamos o desbalanceamento — ponto central do problema.


In [ ]:
plot_class_distribution(df['Class'])


## 4. Distribuição do Valor das Transações


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição geral do Amount
df['Amount'].clip(upper=500).hist(bins=60, ax=axes[0], color='#2196F3', edgecolor='white')
axes[0].set_title('Distribuição do Valor (truncado R$500)', fontsize=13)
axes[0].set_xlabel('Amount')

# Comparação fraude vs legítima
df[df['Class']==0]['Amount'].clip(upper=500).hist(bins=50, ax=axes[1], alpha=0.6, label='Legítima', color='#2196F3')
df[df['Class']==1]['Amount'].clip(upper=500).hist(bins=50, ax=axes[1], alpha=0.8, label='Fraude', color='#F44336')
axes[1].set_title('Legítima vs Fraude', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

print('Valor médio — Legítima:', df[df.Class==0]['Amount'].mean().round(2))
print('Valor médio — Fraude:  ', df[df.Class==1]['Amount'].mean().round(2))


## 5. Padrão Temporal

Time está em segundos. Convertemos para hora para analisar.


In [ ]:
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[df.Class==0]['Hour'].hist(bins=24, ax=axes[0], color='#2196F3', alpha=0.7, label='Legítima')
df[df.Class==1]['Hour'].hist(bins=24, ax=axes[0], color='#F44336', alpha=0.8, label='Fraude')
axes[0].set_title('Transações por Hora do Dia', fontsize=13)
axes[0].set_xlabel('Hora')
axes[0].legend()

# Taxa de fraude por hora
taxa_hora = df.groupby('Hour')['Class'].mean()
taxa_hora.plot(kind='bar', ax=axes[1], color='#E74C3C')
axes[1].set_title('Taxa de Fraude por Hora', fontsize=13)
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Taxa de Fraude')

plt.tight_layout()
plt.show()


## 6. Correlação das Features com Class


In [ ]:
correlacoes = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
correlacoes.head(15).plot(kind='barh', color='#45B7D1', edgecolor='black')
plt.title('Top 15 Features mais Correlacionadas com Fraude', fontsize=13, fontweight='bold')
plt.xlabel('|Correlação|')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 5 features correlacionadas com fraude:')
print(correlacoes.head())


## 7. Exportar Dados Brutos

Salvamos uma cópia local para os próximos notebooks não precisarem redownload.


In [ ]:
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/creditcard.csv', index=False)
print('Dados salvos em data/raw/creditcard.csv')
